<a href="https://colab.research.google.com/github/belleasi/Machine-Learning-models/blob/main/semi-supervised/%20%20self_training_wine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier

np.random.seed(42)

print("=" * 60)
print("SELF-TRAINING - SEMI-SUPERVISED LEARNING")
print("=" * 60)

# Load dataset
wine = load_wine()
X = wine.data
y = wine.target

df = pd.DataFrame(X, columns=wine.feature_names)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nTarget classes:")
print(wine.target_names)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Create partial labels
# -1 means unlabeled in sklearn
y_train_partial = np.full_like(y_train, -1)

# Keep only 20% of training labels
labeled_fraction = 0.20
n_labeled = int(labeled_fraction * len(y_train))

labeled_indices = np.random.choice(
    len(y_train),
    size=n_labeled,
    replace=False
)

y_train_partial[labeled_indices] = y_train[labeled_indices]

print("\nTraining samples:", len(y_train))
print("Labeled samples:", n_labeled)
print("Unlabeled samples:", len(y_train) - n_labeled)

# Supervised-only model
supervised_model = LogisticRegression(max_iter=1000, random_state=42)
supervised_model.fit(
    X_train[labeled_indices],
    y_train[labeled_indices]
)

supervised_preds = supervised_model.predict(X_test)
supervised_acc = accuracy_score(y_test, supervised_preds)

print("\nSupervised-only accuracy:")
print(supervised_acc)

# Semi-supervised self-training model
base_model = LogisticRegression(max_iter=1000, random_state=42)

self_training_model = SelfTrainingClassifier(
    estimator=base_model,
    threshold=0.75,
    max_iter=20
)

self_training_model.fit(X_train, y_train_partial)

semi_preds = self_training_model.predict(X_test)
semi_acc = accuracy_score(y_test, semi_preds)

print("\nSelf-training accuracy:")
print(semi_acc)

print("\nClassification report for self-training:")
print(classification_report(y_test, semi_preds))

# Compare different labeled fractions
fractions = [0.10, 0.20, 0.30, 0.40, 0.50]

supervised_scores = []
semi_scores = []

for frac in fractions:
    y_partial = np.full_like(y_train, -1)
    n_lab = int(frac * len(y_train))

    lab_idx = np.random.choice(
        len(y_train),
        size=n_lab,
        replace=False
    )

    y_partial[lab_idx] = y_train[lab_idx]

    # Supervised only
    sup = LogisticRegression(max_iter=1000, random_state=42)
    sup.fit(X_train[lab_idx], y_train[lab_idx])
    sup_acc = accuracy_score(y_test, sup.predict(X_test))

    # Self-training
    base = LogisticRegression(max_iter=1000, random_state=42)
    semi = SelfTrainingClassifier(
        estimator=base,
        threshold=0.75,
        max_iter=20
    )
    semi.fit(X_train, y_partial)
    semi_acc_loop = accuracy_score(y_test, semi.predict(X_test))

    supervised_scores.append(sup_acc)
    semi_scores.append(semi_acc_loop)

    print(f"{int(frac*100)}% labeled:")
    print(f"  Supervised-only: {sup_acc:.3f}")
    print(f"  Self-training  : {semi_acc_loop:.3f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.plot(
    [f * 100 for f in fractions],
    supervised_scores,
    marker="o",
    label="Supervised Only"
)

plt.plot(
    [f * 100 for f in fractions],
    semi_scores,
    marker="o",
    label="Self-Training"
)

plt.title("Supervised vs Semi-Supervised Learning")
plt.xlabel("Percent of Labeled Training Data")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

# Final summary
print("\nSUMMARY")
print("This experiment compares a supervised model with a semi-supervised self-training model.")
print("The semi-supervised model uses a small amount of labeled data and the remaining unlabeled data.")
print("This belongs in the semi-supervised folder.")